# 🗡️ RPG Class Voice Pack MVP — Qwen3-TTS

This notebook demonstrates how to use the Qwen3-TTS VoiceDesign model to generate a complete set of RPG character voices. We will generate greetings, combat taunts, and death lines for 8 unique classes.

| Class | Description |
|---|---|
| Fighter | Gruff, battle-hardened warrior |
| Wizard | Eccentric, elderly academic |
| Rogue | Sardonic, street-smart thief |
| Cleric | Warm, devoted healer |
| Ranger | Quiet, weathered tracker |
| Paladin | Noble, resonant holy warrior |
| Bard | Theatrical, flamboyant storyteller |
| Necromancer | Cold, softly sinister academic |

In [ ]:
!pip install -q qwen-tts soundfile

import torch
import gc
import os
import numpy as np
import soundfile as sf
from IPython.display import Audio, display
from qwen_tts import Qwen3TTSModel

In [ ]:
OUTPUT_DIR = "/content/rpg_voice_pack"
os.makedirs(OUTPUT_DIR, exist_ok=True)

def clear_vram():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

RPG_CLASSES = {
    "Fighter": {
        "voice_prompt": "A gruff, battle-hardened warrior — rough voice, direct, confident, no-nonsense",
        "greeting": "Name's Garrett. I fight, you pay. Simple as that.",
        "combat_taunt": "Come on then! Is that all you've got?!",
        "death_line": "Tell... my brother... I died well."
    },
    "Wizard": {
        "voice_prompt": "An elderly, eccentric academic — high-pitched, rapid speech, slightly distracted, British",
        "greeting": "Fascinating! A new specimen! I mean — hello, traveler.",
        "combat_taunt": "By the seven moons! Feel the arcane wrath!",
        "death_line": "Ah... I calculated... a seventy percent survival... rate..."
    },
    "Rogue": {
        "voice_prompt": "A smooth, whispering, street-smart thief — sardonic, quick, slight cockney lilt",
        "greeting": "Don't look at me like that. I'm just browsing.",
        "combat_taunt": "You should've checked your pockets first, love.",
        "death_line": "Heh... should've... run faster..."
    },
    "Cleric": {
        "voice_prompt": "A warm, firm, devoted healer — clear diction, gentle authority, compassionate",
        "greeting": "Peace be with you, friend. How may the light serve you?",
        "combat_taunt": "The divine light condemns you! Stand down!",
        "death_line": "Into... your embrace... I return..."
    },
    "Ranger": {
        "voice_prompt": "A quiet, weathered tracker — sparse words, calm, slightly rugged, outdoor-worn voice",
        "greeting": "You're louder than you think. I heard you from the ridge.",
        "combat_taunt": "Stay still. This will be cleaner if you don't move.",
        "death_line": "The forest... will remember..."
    },
    "Paladin": {
        "voice_prompt": "A noble, resonant, idealistic holy warrior — strong, ceremonial, slightly formal",
        "greeting": "Well met! I am Sir Aldric, sworn to defend the innocent.",
        "combat_taunt": "For justice! For the kingdom! FOR THE LIGHT!",
        "death_line": "I... fulfilled my oath... that is enough."
    },
    "Bard": {
        "voice_prompt": "A theatrical, melodic, flamboyant storyteller — expressive, warm, slightly overdramatic",
        "greeting": "Ah, what a tale this encounter shall make! The bard Flinn, at your service!",
        "combat_taunt": "Verse one: the villain charged. Verse two: they fell. Short poem!",
        "death_line": "Every great... story needs... a tragic hero... remember me!"
    },
    "Necromancer": {
        "voice_prompt": "A cold, deliberate, softly sinister academic — unhurried, precise, faintly amused by death",
        "greeting": "Ah. The living. How... refreshingly temporary you all are.",
        "combat_taunt": "Rise. Kill them. Then we'll have a proper conversation.",
        "death_line": "Death and I... are old... friends..."
    }
}

In [ ]:
clear_vram()

model_id = "Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign"
print(f"Loading model: {model_id}")

model = Qwen3TTSModel.from_pretrained(
    model_id, 
    device_map="cuda:0", 
    dtype=torch.bfloat16, 
    attn_implementation="sdpa"
)

print("Model loaded successfully!")

In [ ]:
def generate_and_save(text, instruct, filename):
    res = model.generate_voice_design(text, "English", instruct)
    
    # Handle different possible return formats
    if isinstance(res, tuple):
        audio, sr = res
    else:
        audio, sr = res, 24000
        
    if hasattr(audio, 'cpu'):
        audio = audio.cpu().numpy()
        
    audio = np.squeeze(audio)
    filepath = os.path.join(OUTPUT_DIR, filename)
    sf.write(filepath, audio, sr)
    
    duration = len(audio) / sr
    return filepath, audio, sr, duration

stats = []
greetings_audio = []
global_sr = 24000
generated_paths = []

for cls, data in RPG_CLASSES.items():
    print("="*50)
    print(f"--- 🗡️ Voice Card: {cls} ---")
    print(f"Prompt: {data['voice_prompt']}\n")
    
    # Greeting
    print("🗣️ Greeting:", data['greeting'])
    path_g, aud_g, sr, dur_g = generate_and_save(data['greeting'], data['voice_prompt'], f"rpg_{cls.lower()}_greeting.wav")
    generated_paths.append(path_g)
    greetings_audio.append(aud_g)
    global_sr = sr
    display(Audio(path_g))
    
    # Combat Taunt
    print("⚔️ Combat Taunt:", data['combat_taunt'])
    path_c, _, _, dur_c = generate_and_save(data['combat_taunt'], data['voice_prompt'], f"rpg_{cls.lower()}_combat.wav")
    generated_paths.append(path_c)
    display(Audio(path_c))
    
    # Death Line
    print("💀 Death Line:", data['death_line'])
    path_d, _, _, dur_d = generate_and_save(data['death_line'], data['voice_prompt'], f"rpg_{cls.lower()}_death.wav")
    generated_paths.append(path_d)
    display(Audio(path_d))
    
    stats.append({
        "Class": cls,
        "Greeting": dur_g,
        "Combat": dur_c,
        "Death": dur_d,
        "Total": dur_g + dur_c + dur_d
    })
    print("\n")

In [ ]:
print("🎬 Generating Party Introduction Scene...")
silence = np.zeros(int(0.3 * global_sr))
scene_parts = []

for audio in greetings_audio:
    scene_parts.append(audio)
    scene_parts.append(silence)
    
full_scene_audio = np.concatenate(scene_parts)
scene_path = os.path.join(OUTPUT_DIR, "rpg_full_party_intro.wav")
sf.write(scene_path, full_scene_audio, global_sr)

print("Full Party Intro Scene (0.3s silence between greetings):")
display(Audio(scene_path))

In [ ]:
print("📊 Stats Summary:")
print(f"{'Class':<12} | {'Greeting':<10} | {'Combat':<10} | {'Death':<10} | {'Total'}")
print("-"*65)
for s in stats:
    print(f"{s['Class']:<12} | {s['Greeting']:.2f}s     | {s['Combat']:.2f}s     | {s['Death']:.2f}s     | {s['Total']:.2f}s")

In [ ]:
import shutil
from google.colab import files

print("📦 Zipping and downloading the voice pack...")
shutil.make_archive(OUTPUT_DIR, 'zip', OUTPUT_DIR)
files.download(f"{OUTPUT_DIR}.zip")